# JEPA Talking-Head: Multi-Identity Diffusion Decoder Training

**Purpose**: test the data-scale hypothesis directly. Every prior architecture/loss
experiment on the single-identity (`May`) dataset plateaued around chance-level
mouth-region matching accuracy, and a literature review found every comparable SOTA
system trains on multi-speaker corpora 400-35,000x larger than our ~6,000-frame
single-identity set. This notebook trains the SAME decoder recipe (no cross-attention,
no contrastive loss - both paused per the architecture-work pause decision) on the
newly-built 11-identity dataset instead, to see whether more data alone moves the
needle before any further architecture changes are considered.

**Validation design**: 2 of the 11 identities (`AnnMarieBuerkle`, `DebFischer0` - clean
framing, moderate size, ~14% of total frames) are held out ENTIRELY from training.
The decisive check at the end evaluates separately on:
- a small in-distribution validation split from the 9 TRAINING identities (has the
  model fit the training distribution at all?)
- the 2 fully held-out identities (does it generalize to an unseen face + voice? -
  this is the real test of the data-scale hypothesis)

**Architecture**: identical to the validated `train_diffusion_decoder` v5 recipe
(`GenerationLatentAdapter` + `PatchTokenDiffusionDecoder` + SyncNet perceptual loss +
SIGReg, 2-phase curriculum). `use_cross_attention=False`, `contrastive_loss_weight=0.0`
throughout - see `JEPA_TalkingHead_GenLatent.ipynb` for why those were paused (Phase 5
result) and the KB note for the full history.

**Before running the training section**: make sure the Drive upload of `data/` has
finished for every identity listed below - there's a readiness check cell for this.


## 1. Setup

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

CODE_BASE = '/content/drive/MyDrive/Data/le-wm/'

# Unchanged from JEPA_TalkingHead_GenLatent.ipynb - the original, already-validated
# single-identity dataset. Kept at its existing path rather than pointing at
# data/May (the newly-reprocessed copy under CODE_BASE) to avoid any risk of the two
# not being byte-identical; this is the copy every prior phase's results are based on.
REAL_DATA_PATH = '/content/drive/MyDrive/Data/TalkingHead/May_extracted'

# Output of data_utils_multi_identity.py - one subfolder per HDTF identity.
NEW_DATA_ROOT = os.path.join(CODE_BASE, 'data')

# 2 identities held out ENTIRELY from training for a genuine generalization test -
# chosen for clean framing (close to May's tight-headshot convention, confirmed via
# contact-sheet inspection) and moderate size (~14% of total frames combined, not
# so large that holding them out starves training, not so small the validation
# signal would be noisy).
HELD_OUT_NAMES = ["AnnMarieBuerkle", "DebFischer0"]

ALL_NEW_NAMES = [
    "AnnMarieBuerkle", "AustinScott0", "BillCassidy0", "BobbySchilling", "CandiceMiller",
    "CarlyFiorina", "ChuckGrassley", "DanNewhouse", "DaveCamp", "DebFischer0",
]
TRAIN_NEW_NAMES = [n for n in ALL_NEW_NAMES if n not in HELD_OUT_NAMES]

TRAIN_IDENTITIES = {"May": REAL_DATA_PATH}
TRAIN_IDENTITIES.update({n: os.path.join(NEW_DATA_ROOT, n) for n in TRAIN_NEW_NAMES})

HELD_OUT_IDENTITIES = {n: os.path.join(NEW_DATA_ROOT, n) for n in HELD_OUT_NAMES}

print(f"Train identities ({len(TRAIN_IDENTITIES)}): {list(TRAIN_IDENTITIES.keys())}")
print(f"Held-out identities ({len(HELD_OUT_IDENTITIES)}): {list(HELD_OUT_IDENTITIES.keys())}")

# Point this at whichever trained-encoder checkpoint you want to build on (the
# frozen JEPA encoder - not retrained here). Set to None to auto-detect the most
# recently modified checkpoint under checkpoints/ (unreliable after Drive sync
# resets mtimes - prefer setting this explicitly once you know which run you want).
ENCODER_CHECKPOINT_PATH = None


In [ ]:
# Data readiness check - run this any time to see which identities are ready
# (useful while data/ is still uploading to Drive in the background).
def _check_identity_ready(name, path):
    gt_dir = os.path.join(path, "gt_imgs")
    ave_path = os.path.join(path, "aud_ave.npy")
    n_gt = len(os.listdir(gt_dir)) if os.path.isdir(gt_dir) else 0
    ready = n_gt > 0 and os.path.isfile(ave_path)
    return ready, n_gt


print("Data readiness check:")
_all_ready = True
for _name, _path in {**TRAIN_IDENTITIES, **HELD_OUT_IDENTITIES}.items():
    _ready, _n_gt = _check_identity_ready(_name, _path)
    _all_ready = _all_ready and _ready
    print(f"  [{'OK' if _ready else 'MISSING':7s}] {_name:20s} n_gt_imgs={_n_gt}")

if _all_ready:
    print("\nAll identities ready - safe to run the Training section.")
else:
    print("\nNot all identities are ready yet (Drive upload still in progress?). "
          "Setup/Model-definition cells below don't need the data and are safe to "
          "run now; hold off on the Training section until every identity shows OK.")


In [ ]:
# Install uv (if not already installed)
!pip install uv

# Install system dependencies required for building some Python packages (e.g., swig for box2d-py)
!apt-get update -qq > /dev/null
!apt-get install -qq -y swig > /dev/null

# Install stable-worldmodel and its dependencies using uv
!uv pip install stable-worldmodel[train,env] hydra-core torchview graphviz
!pip install -q diffusers


In [ ]:
os.chdir(CODE_BASE)
print(f"Changed current working directory to: {os.getcwd()}")

import sys
if CODE_BASE not in sys.path:
    sys.path.insert(0, CODE_BASE)
print(f"Added {CODE_BASE} to sys.path")

import math
import glob
import gc
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, ConcatDataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from einops import rearrange
from torchvision.models.vision_transformer import vit_b_16
from tqdm import tqdm
from omegaconf import OmegaConf, DictConfig

from diffusers import DDPMScheduler, DDIMScheduler

from jepa import JEPA
from module import ARPredictor, MLP, SIGReg, Transformer, ConditionalBlock, modulate

print("Imports OK.")


## 2. Model definitions (causal-masking fix + JEPA classes)

In [ ]:
# --- Causal-masking fix (carried over from JEPA_TalkingHead_GenLatent.ipynb) ---
# ConditionalBlock defaults causal=True, which is correct for jepa.py's own
# incremental rollout() but wrong for both the ARPredictor's fixed, fully-
# known-context parallel prediction and the diffusion decoder's Transformer
# (patch tokens / generation-latent conditioning have no causal/temporal
# order). Self-contained (imports its own deps) so it doesn't depend on
# where in the notebook it runs.
from module import ConditionalBlock, modulate


def _conditional_block_forward_causal_aware(self, x, c):
    shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
        self.adaLN_modulation(c).chunk(6, dim=-1)
    )
    causal = getattr(self, "causal", True)  # default True = unchanged existing behavior
    x = x + gate_msa * self.attn(modulate(self.norm1(x), shift_msa, scale_msa), causal=causal)
    x = x + gate_mlp * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))
    return x


ConditionalBlock.forward = _conditional_block_forward_causal_aware


# --- Core JEPA model definitions (carried over, unchanged) ---

class AudioFeatureEmbedder(nn.Module):
    def __init__(self, audio_feature_dim, emb_dim, hidden_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(audio_feature_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, emb_dim)
        )

    def forward(self, audio_features):
        return self.net(audio_features)


class FaceEncoder(nn.Module):
    def __init__(self, image_size=224, embed_dim=768):
        super().__init__()
        self.vit = vit_b_16(weights=None)
        self.vit.heads = nn.Identity()
        self.embed_dim = embed_dim

    def forward(self, pixels, **kwargs):
        if len(pixels.shape) == 5:
            pixels = rearrange(pixels, 'b t c h w -> (b t) c h w')

        x = self.vit._process_input(pixels)
        n = x.shape[0]

        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = self.vit.encoder(x)

        class Output:
            pass
        out = Output()
        out.last_hidden_state = x
        return out


class TalkingHeadJEPA(nn.Module):
    def __init__(
        self,
        image_size=224,
        audio_feature_dim=512,
        embed_dim=256,
        hidden_dim=256,
        history_size=4,
        audio_expert=None,
        freeze_audio_expert_params=False
    ):
        super().__init__()
        self.history_size = history_size
        self.embed_dim = embed_dim

        self.face_encoder = FaceEncoder(image_size=image_size, embed_dim=embed_dim)

        if not audio_expert:
            self.audio_embedder = AudioFeatureEmbedder(audio_feature_dim=audio_feature_dim, emb_dim=embed_dim)
        else:
            self.audio_embedder = audio_expert
            if freeze_audio_expert_params:
                for param in self.audio_embedder.parameters():
                    param.requires_grad = False

        self.predictor = ARPredictor(
            num_frames=history_size,
            input_dim=embed_dim,
            hidden_dim=hidden_dim,
            output_dim=embed_dim,
            depth=6,
            heads=8,
            mlp_dim=hidden_dim * 4,
        )
        for _block in self.predictor.transformer.layers:
            _block.causal = False

        self.projector = MLP(input_dim=768, output_dim=embed_dim, hidden_dim=1024, norm_fn=nn.BatchNorm1d)
        self.pred_proj = MLP(input_dim=embed_dim, output_dim=embed_dim, hidden_dim=1024, norm_fn=nn.BatchNorm1d)

        self.jepa = JEPA(
            encoder=self.face_encoder,
            predictor=self.predictor,
            action_encoder=self.audio_embedder,
            projector=self.projector,
            pred_proj=self.pred_proj,
        )

        self.sigreg = SIGReg()
        self.sigreg_weight = 1.0

    def encode(self, batch):
        encoder_out = self.face_encoder(batch["pixels"], interpolate_pos_encoding=True)
        raw_features = encoder_out.last_hidden_state

        if len(raw_features.shape) == 3:
            raw_features = raw_features[:, 0]

        emb = self.projector(raw_features)
        act_emb = self.audio_embedder(batch["action"])

        return {"emb": emb, "act_emb": act_emb}


class ResBlock(nn.Module):
    """Reused by PatchGridToImageDecoder below."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.LeakyReLU(0.2, inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        return x + self.conv2(self.relu(self.bn1(self.conv1(x))))


print("Model definitions loaded (causal-masking fix applied, JEPA classes, ResBlock).")


## 3. Load the frozen, already-trained JEPA encoder

In [ ]:
# --- Load the frozen, already-trained JEPA encoder ---
# This notebook does NOT retrain the encoder. It builds on whatever checkpoint you
# already have, frozen, and trains only the decoder side (generation-latent adapter +
# diffusion decoder) on the multi-identity dataset.

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

clear_gpu()

CONFIG_FILE_PATH = os.path.join(CODE_BASE, 'config', 'train', 'talking_head.yaml')
cfg = OmegaConf.merge(
    OmegaConf.load(CONFIG_FILE_PATH),
    OmegaConf.create({"model": {"audio_feature_dim": 512}}),  # must match the checkpoint's training config
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_frozen_encoder(cfg, checkpoint_path=None, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if checkpoint_path is None:
        candidates = glob.glob(os.path.join("checkpoints", "*.pth"))
        if not candidates:
            raise FileNotFoundError(
                "No checkpoints found in 'checkpoints/'. Set ENCODER_CHECKPOINT_PATH explicitly, "
                "or train an encoder first in JEPA_TalkingHead_Experiments.ipynb."
            )
        checkpoint_path = max(candidates, key=os.path.getctime)
        print("ENCODER_CHECKPOINT_PATH was None - auto-detected latest by mtime "
              "(unreliable after Drive sync; verify this is the checkpoint you want):")
    print(f"Loading frozen encoder from: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    encoder_model = TalkingHeadJEPA(
        image_size=cfg.model.image_size,
        audio_feature_dim=cfg.model.audio_feature_dim,
        embed_dim=cfg.model.embed_dim,
        hidden_dim=cfg.model.hidden_dim,
        history_size=cfg.model.history_size,
    ).to(device)
    encoder_model.load_state_dict(checkpoint["model_state_dict"])

    # Frozen throughout - no gradients, eval() mode (BatchNorm1d in
    # model.jepa.projector uses running stats only in eval mode; torch.no_grad()
    # alone does not stop it from updating running stats in train() mode).
    for p in encoder_model.parameters():
        p.requires_grad = False
    encoder_model.eval()

    print(f"Loaded (encoder trained through epoch {checkpoint.get('epoch', '?')}). Frozen, eval() mode.")
    return encoder_model


model = load_frozen_encoder(cfg, checkpoint_path=ENCODER_CHECKPOINT_PATH, device=device)


## 4. Dataset

In [ ]:
class SyncTalkDataset(Dataset):
    # audio_window_radius default 5 (not 0) - every SOTA talking-head system uses a
    # WINDOWED sequence of audio frames, not a single per-frame average (coarticulation:
    # the mouth starts moving before the phoneme).
    def __init__(self, data_root, seq_len=8, image_size=224, audio_window_radius=5):
        self.data_root = Path(data_root)
        self.seq_len = seq_len
        self.image_size = image_size
        self.audio_window_radius = audio_window_radius

        # CRITICAL: plain sorted() on filenames like "0.jpg","1.jpg","10.jpg","100.jpg"
        # sorts them as STRINGS, not numbers - sort by the integer frame number instead.
        self.image_paths = sorted((self.data_root / 'gt_imgs').glob('*.jpg'), key=lambda p: int(p.stem))
        if not self.image_paths:
            self.image_paths = sorted((self.data_root / 'ori_imgs').glob('*.jpg'), key=lambda p: int(p.stem))

        audio_path = self.data_root / 'aud_ave.npy'
        if audio_path.exists():
            self.audio_features = np.load(audio_path)
        else:
            print(f"Warning: {audio_path} not found, using dummy audio")
            self.audio_features = np.zeros((len(self.image_paths), 1024))

    def __len__(self):
        return max(0, len(self.image_paths) - self.seq_len)

    def _audio_window(self, frame_idx):
        """+/- audio_window_radius frames of audio context around frame_idx, edge-clamped
        (repeats the boundary frame rather than zero-padding). Returns a flat vector of
        length (2*radius+1)*base_audio_dim."""
        n = len(self.audio_features)
        r = self.audio_window_radius
        lo, hi = max(0, frame_idx - r), min(n, frame_idx + r + 1)
        window = self.audio_features[lo:hi]
        needed = 2 * r + 1
        if window.shape[0] < needed:
            if lo == 0:
                pad = np.repeat(window[0:1], needed - window.shape[0], axis=0)
                window = np.concatenate([pad, window], axis=0)
            else:
                pad = np.repeat(window[-1:], needed - window.shape[0], axis=0)
                window = np.concatenate([window, pad], axis=0)
        return window.reshape(-1)

    def __getitem__(self, idx):
        frames = []
        for i in range(idx, idx + self.seq_len):
            img = cv2.imread(str(self.image_paths[i]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.image_size, self.image_size))
            frames.append(img)

        frames = np.stack(frames).astype(np.float32) / 255.0
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2)  # (T, C, H, W)

        audio_idx = min(idx, len(self.audio_features) - self.seq_len)
        windows = [self._audio_window(audio_idx + t) for t in range(self.seq_len)]
        audio = torch.from_numpy(np.stack(windows)).float()  # (T, (2*radius+1)*base_audio_dim)

        return frames, audio


def build_concat_dataset(identity_paths, seq_len, audio_window_radius=5, verbose=True):
    """identity_paths: dict {name: data_root}. Returns a single ConcatDataset spanning
    every identity, plus the per-identity sequence counts for logging."""
    datasets, counts = [], {}
    for name, path in identity_paths.items():
        ds = SyncTalkDataset(data_root=path, seq_len=seq_len, audio_window_radius=audio_window_radius)
        counts[name] = len(ds)
        if verbose:
            print(f"  {name:20s} {len(ds):>6} sequences")
        datasets.append(ds)
    return ConcatDataset(datasets), counts


print("SyncTalkDataset + build_concat_dataset() defined.")


## 5. Frozen pretrained lip-sync expert (Wav2Lip SyncNet)

In [ ]:
# --- Wav2Lip SyncNet (Prajwal et al. 2020, arXiv:2008.10010) ---
# Architecture copied verbatim from github.com/Rudrabha/Wav2Lip
# (models/syncnet.py, models/conv.py) so the pretrained checkpoint's
# state_dict keys match exactly.
#
# Role in this notebook: a frozen, pretrained mouth/expression feature extractor used
# as a PERCEPTUAL AUXILIARY LOSS on the decoder's reconstruction path (face_encoder
# branch only) - not as generation-time conditioning (its face branch needs an actual
# face image, unavailable when generating a genuinely new/predicted frame; its audio
# branch expects raw mel-spectrogram windows our precomputed aud_ave.npy isn't in).

class Conv2d(nn.Module):
    def __init__(self, cin, cout, kernel_size, stride, padding, residual=False, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conv_block = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size, stride, padding),
            nn.BatchNorm2d(cout)
        )
        self.act = nn.ReLU()
        self.residual = residual

    def forward(self, x):
        out = self.conv_block(x)
        if self.residual:
            out += x
        return self.act(out)


class SyncNet_color(nn.Module):
    def __init__(self):
        super().__init__()

        self.face_encoder = nn.Sequential(
            Conv2d(15, 32, kernel_size=(7, 7), stride=1, padding=3),

            Conv2d(32, 64, kernel_size=5, stride=(1, 2), padding=1),
            Conv2d(64, 64, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(64, 64, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            Conv2d(128, 128, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(128, 128, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(128, 128, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            Conv2d(256, 256, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(256, 256, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
            Conv2d(512, 512, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(512, 512, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(512, 512, kernel_size=3, stride=2, padding=1),
            Conv2d(512, 512, kernel_size=3, stride=1, padding=0),
            Conv2d(512, 512, kernel_size=1, stride=1, padding=0),
        )

        self.audio_encoder = nn.Sequential(
            Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            Conv2d(32, 32, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(32, 32, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(32, 64, kernel_size=3, stride=(3, 1), padding=1),
            Conv2d(64, 64, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(64, 64, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(64, 128, kernel_size=3, stride=3, padding=1),
            Conv2d(128, 128, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(128, 128, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(128, 256, kernel_size=3, stride=(3, 2), padding=1),
            Conv2d(256, 256, kernel_size=3, stride=1, padding=1, residual=True),
            Conv2d(256, 256, kernel_size=3, stride=1, padding=1, residual=True),

            Conv2d(256, 512, kernel_size=3, stride=1, padding=0),
            Conv2d(512, 512, kernel_size=1, stride=1, padding=0),
        )

    def forward(self, audio_sequences, face_sequences):
        face_embedding = self.face_encoder(face_sequences)
        audio_embedding = self.audio_encoder(audio_sequences)

        audio_embedding = audio_embedding.view(audio_embedding.size(0), -1)
        face_embedding = face_embedding.view(face_embedding.size(0), -1)

        audio_embedding = F.normalize(audio_embedding, p=2, dim=1)
        face_embedding = F.normalize(face_embedding, p=2, dim=1)

        return audio_embedding, face_embedding


SYNCNET_CKPT_PATH = "lipsync_expert.pth"
SYNCNET_CKPT_URL = "https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/lipsync_expert.pth?download=true"

if not os.path.exists(SYNCNET_CKPT_PATH):
    print(f"Downloading SyncNet checkpoint from {SYNCNET_CKPT_URL} ...")
    os.system(f'wget -q -O "{SYNCNET_CKPT_PATH}" "{SYNCNET_CKPT_URL}"')

syncnet = SyncNet_color().to(device)

_raw_ckpt = torch.load(SYNCNET_CKPT_PATH, map_location=device)
_state_dict = _raw_ckpt.get("state_dict", _raw_ckpt)
# Wav2Lip's released checkpoints were saved from a DataParallel-wrapped
# model, so keys are prefixed with "module." - strip it if present.
_state_dict = {k.replace("module.", "", 1) if k.startswith("module.") else k: v
               for k, v in _state_dict.items()}
_missing, _unexpected = syncnet.load_state_dict(_state_dict, strict=False)
print(f"SyncNet checkpoint loaded. missing keys: {len(_missing)}, unexpected keys: {len(_unexpected)}")

for p in syncnet.parameters():
    p.requires_grad = False
syncnet.eval()
print("SyncNet frozen, eval() mode.")


@torch.no_grad()
def make_mouth_window(frames_btchw, num_frames=5, size=96):
    """
    frames_btchw: (B, T, 3, H, W) in [0, 1], T >= num_frames.
    Crops the lower half of each frame (mouth region, matching Wav2Lip's own
    preprocessing convention), resizes to size x size, takes the first
    num_frames consecutive frames, and concatenates along the channel dim
    -> (B, 3*num_frames, size, size), SyncNet's expected face_sequences input shape.
    """
    B, T, C, H, W = frames_btchw.shape
    assert T >= num_frames, f"need at least {num_frames} consecutive frames, got {T}"
    window = frames_btchw[:, :num_frames]                      # (B, num_frames, 3, H, W)
    mouth = window[:, :, :, H // 2:, :]                         # lower half
    mouth = rearrange(mouth, "b t c h w -> (b t) c h w")
    mouth = F.interpolate(mouth, size=(size, size), mode="bilinear", align_corners=False)
    mouth = rearrange(mouth, "(b t) c h w -> b (t c) h w", b=B, t=num_frames)
    return mouth


def syncnet_perceptual_loss(decoded_window_btchw, gt_window_btchw, num_frames=5):
    """Cosine-distance loss between SyncNet's face_encoder embedding of a decoded
    5-frame mouth window vs. the ground-truth window."""
    decoded_mouth = make_mouth_window(decoded_window_btchw, num_frames=num_frames)
    with torch.no_grad():
        gt_mouth = make_mouth_window(gt_window_btchw, num_frames=num_frames)
    decoded_mouth = rearrange(decoded_mouth, "b c h w -> b c h w")  # no-op, keeps grad graph explicit
    _, decoded_embed = syncnet(torch.zeros(decoded_mouth.size(0), 1, 80, 16, device=decoded_mouth.device),
                                decoded_mouth)
    with torch.no_grad():
        _, gt_embed = syncnet(torch.zeros(gt_mouth.size(0), 1, 80, 16, device=gt_mouth.device), gt_mouth)
    return (1 - F.cosine_similarity(decoded_embed, gt_embed, dim=-1)).mean()


print("syncnet_perceptual_loss() defined.")


## 6. Diffusion decoder architecture + generation-latent adapter

In [ ]:
# --- Diffusion decoder backbone + generation-latent adapter ---
# Carried over verbatim from JEPA_TalkingHead_GenLatent.ipynb. Cross-attention
# (AudioCrossAttention/audio_seq) and the contrastive loss (MouthContentProjector/
# audio_content_contrastive_loss) are DEFINED here for compatibility with any
# checkpoint that used them, but this notebook's training runs keep
# use_cross_attention=False and contrastive_loss_weight=0.0 - both were found to hurt
# results on the single-identity dataset (Phase 5), most likely because a single
# speaker's frames don't give the contrastive loss real negative diversity. Multi-
# identity batches may resolve that weakness - worth revisiting once this baseline
# (data-scale-only) run has a result, not before.

class TimestepEmbedding(nn.Module):
    def __init__(self, dim, freq_dim=256):
        super().__init__()
        self.freq_dim = freq_dim
        self.mlp = nn.Sequential(nn.Linear(freq_dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        half = self.freq_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None]
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return self.mlp(emb)


class GenerationLatentAdapter(nn.Module):
    """VASA-1-style disentangled generation latent: a small trainable adapter off the
    (frozen) JEPA representation. Combines a fresh, trainable projection of RAW
    audio_features with pooled_state."""
    def __init__(self, embed_dim=256, audio_feature_dim=512, gen_dim=256, hidden_dim=512):
        super().__init__()
        self.audio_proj = nn.Linear(audio_feature_dim, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, gen_dim),
        )
        self.null_cond = nn.Parameter(torch.zeros(gen_dim))  # CFG-style unconditional embedding
        self.null_pooled_state = nn.Parameter(torch.zeros(embed_dim))

    def forward(self, pooled_state, raw_audio):
        audio_embed = self.audio_proj(raw_audio)
        return self.net(torch.cat([pooled_state, audio_embed], dim=-1))


class AudioCrossAttention(nn.Module):
    """Real cross-attention between patch tokens (queries) and a SEQUENCE of per-frame
    raw audio features (keys/values). to_out is zero-initialized so this starts as an
    exact no-op. Not used in this notebook's default config (use_cross_attention=False)."""
    def __init__(self, dim, kv_dim, heads=4, dim_head=32):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.norm_q = nn.LayerNorm(dim)
        self.to_q = nn.Linear(dim, inner_dim, bias=False)
        self.to_kv = nn.Linear(kv_dim, inner_dim * 2, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)
        nn.init.zeros_(self.to_out.weight)
        nn.init.zeros_(self.to_out.bias)

    def forward(self, x, audio_seq):
        """x: (B, N_patches, dim) queries. audio_seq: (B, T_audio, kv_dim) keys/values."""
        q = self.to_q(self.norm_q(x))
        k, v = self.to_kv(audio_seq).chunk(2, dim=-1)
        q = rearrange(q, "b n (h d) -> b h n d", h=self.heads)
        k = rearrange(k, "b t (h d) -> b h t d", h=self.heads)
        v = rearrange(v, "b t (h d) -> b h t d", h=self.heads)
        out = F.scaled_dot_product_attention(q, k, v)
        out = rearrange(out, "b h n d -> b n (h d)")
        return self.to_out(out)


class PatchTokenDiffusionDecoder(nn.Module):
    """DiT-style denoiser operating on the ViT patch-token grid, conditioned on the
    GenerationLatentAdapter's output + diffusion timestep, via AdaLN-zero
    ConditionalBlock. Includes a learned per-patch spatial gate for audio (mouth-region
    prior) and optional cross-attention layers (unused by default - see module docstring
    above)."""
    def __init__(self, patch_dim=128, num_patches=196, cond_dim=256, audio_embed_dim=256,
                 raw_audio_feature_dim=512, depth=4, heads=4, mlp_dim=512, grid=14):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches, patch_dim) * 0.02)
        self.time_embed = TimestepEmbedding(cond_dim)
        self.cond_proj = nn.Linear(cond_dim, patch_dim)

        self.audio_spatial_proj = nn.Linear(audio_embed_dim, patch_dim)
        mouth_patch_start = (grid // 2) * grid  # bottom half - matches MOUTH_PATCH_START used below
        gate_init = torch.full((num_patches,), -2.2)   # sigmoid(-2.2) ~= 0.10 - mostly off elsewhere
        gate_init[mouth_patch_start:] = 2.2             # sigmoid(2.2)  ~= 0.90 - mostly on for mouth patches
        self.audio_gate_logit = nn.Parameter(gate_init)  # learnable, but starts from this prior

        self.transformer = Transformer(
            input_dim=patch_dim,
            hidden_dim=patch_dim,
            output_dim=patch_dim,
            depth=depth,
            heads=heads,
            dim_head=max(patch_dim // heads, 8),
            mlp_dim=mlp_dim,
            block_class=ConditionalBlock,
        )
        for _block in self.transformer.layers:
            _block.causal = False

        self.cross_attn_layers = nn.ModuleList([
            AudioCrossAttention(dim=patch_dim, kv_dim=raw_audio_feature_dim, heads=heads)
            for _ in range(depth)
        ])

    def forward(self, noisy_patches, timesteps, cond_vec, audio_embed=None, audio_seq=None):
        """
        cond_vec: (B, cond_dim) - applied uniformly to every patch.
        audio_embed: optional (B, audio_embed_dim) - spatially gated (mouth-region prior).
        audio_seq: optional (B, T_audio, raw_audio_feature_dim) - cross-attention keys/
                  values. None => behavior identical to no cross-attention at all.
        """
        t_emb = self.time_embed(timesteps)
        c = self.cond_proj(cond_vec + t_emb)
        c = c.unsqueeze(1).expand(-1, noisy_patches.size(1), -1)
        if audio_embed is not None:
            audio_c = self.audio_spatial_proj(audio_embed)              # (B, patch_dim)
            gate = torch.sigmoid(self.audio_gate_logit).view(1, -1, 1)  # (1, num_patches, 1)
            c = c + gate * audio_c.unsqueeze(1)
        x = noisy_patches + self.pos_embedding

        if audio_seq is None:
            return self.transformer(x, c)

        x = self.transformer.input_proj(x)
        c_t = self.transformer.cond_proj(c)
        for block, cross_attn in zip(self.transformer.layers, self.cross_attn_layers):
            x = block(x, c_t)
            x = x + cross_attn(x, audio_seq)
        x = self.transformer.norm(x)
        return self.transformer.output_proj(x)


class PatchGridToImageDecoder(nn.Module):
    """Decodes a denoised (B, 196, patch_dim) grid back to a 224x224 RGB image."""
    def __init__(self, patch_dim=128, grid=14):
        super().__init__()
        self.grid = grid
        self.proj = nn.Linear(patch_dim, 256)
        self.deconv = nn.Sequential(
            ResBlock(256),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 28x28
            nn.Conv2d(256, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            ResBlock(128),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 56x56
            nn.Conv2d(128, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            ResBlock(64),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 112x112
            nn.Conv2d(64, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 224x224
            nn.Conv2d(32, 3, kernel_size=3, padding=1), nn.Sigmoid(),
        )

    def forward(self, patch_grid):
        b, n, d = patch_grid.shape
        x = self.proj(patch_grid)
        x = x.transpose(1, 2).reshape(b, 256, self.grid, self.grid)
        return self.deconv(x)


class MouthContentProjector(nn.Module):
    """Projects TRUE mouth-region patch tokens into gen_latent_adapter.audio_proj's
    embedding space, for the (currently unused) contrastive loss below."""
    def __init__(self, patch_dim=128, contrastive_dim=128):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(patch_dim, contrastive_dim), nn.GELU(), nn.Linear(contrastive_dim, contrastive_dim)
        )

    def forward(self, mouth_patch_tok):
        """mouth_patch_tok: (B, num_mouth_patches, patch_dim) -> (B, contrastive_dim)"""
        return self.proj(mouth_patch_tok.mean(dim=1))


def audio_content_contrastive_loss(mouth_embed, audio_embed, temperature=0.1):
    """Symmetric InfoNCE: each sample's true mouth content should be more similar to
    its OWN audio embedding than to any other sample's, in-batch. Not used by default
    in this notebook (contrastive_loss_weight=0.0) - see module docstring above."""
    mouth_n = F.normalize(mouth_embed, dim=-1)
    audio_n = F.normalize(audio_embed, dim=-1)
    logits = mouth_n @ audio_n.T / temperature  # (B, B)
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_m2a = F.cross_entropy(logits, labels)
    loss_a2m = F.cross_entropy(logits.T, labels)
    return (loss_m2a + loss_a2m) / 2


print("Decoder architecture defined (PatchTokenDiffusionDecoder, GenerationLatentAdapter, "
      "PatchGridToImageDecoder; cross-attention + contrastive loss defined but OFF by default).")


In [ ]:
import torchvision


class VGGPerceptualLoss(nn.Module):
    """Frozen VGG16 perceptual loss (relu1_2/relu2_2/relu3_3/relu4_3 feature L1),
    instead of raw pixel MSE alone. Frozen throughout."""
    def __init__(self, layer_ids=(3, 8, 15, 22), device="cuda"):
        super().__init__()
        weights = torchvision.models.VGG16_Weights.IMAGENET1K_V1
        vgg = torchvision.models.vgg16(weights=weights).features.to(device).eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.layer_ids = set(layer_ids)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _normalize(self, x):
        return (x - self.mean) / self.std

    def _features(self, x):
        x = self._normalize(x)
        feats = []
        max_layer = max(self.layer_ids)
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            if i in self.layer_ids:
                feats.append(x)
            if i == max_layer:
                break
        return feats

    def forward(self, decoded, gt, size=112):
        """decoded: (B, 3, H, W) in [0, 1], keeps grad. gt: (B, 3, H, W) in [0, 1], no grad needed."""
        decoded = F.interpolate(decoded, size=(size, size), mode="bilinear", align_corners=False)
        with torch.no_grad():
            gt = F.interpolate(gt, size=(size, size), mode="bilinear", align_corners=False)
        decoded_feats = self._features(decoded)
        with torch.no_grad():
            gt_feats = self._features(gt)
        loss = 0.0
        for df, gf in zip(decoded_feats, gt_feats):
            loss = loss + F.l1_loss(df, gf)
        return loss / len(decoded_feats)


print("VGGPerceptualLoss defined.")


## 7. Training loop (multi-identity `ConcatDataset`, curriculum)

Only change from `JEPA_TalkingHead_GenLatent.ipynb`'s `train_diffusion_decoder`: the
dataset is built from `train_data_roots` (a `{name: path}` dict, via
`build_concat_dataset` above) instead of a single hardcoded `REAL_DATA_PATH`. Everything
else - the encode/loss/checkpoint logic - is unchanged. The `val_frac` split here is a
small **in-distribution** slice of the training identities (for per-epoch monitoring
only) - it does NOT include the 2 fully held-out identities, which are never touched by
this function at all and are evaluated separately in the decisive-check section below.

In [ ]:
# --- Diffusion decoder training loop, v6: multi-identity ConcatDataset ---
import os


def train_diffusion_decoder(
    model,
    cfg,
    train_data_roots,
    num_epochs=10,
    batch_size=32,
    val_frac=0.05,
    patch_dim=128,
    gen_dim=256,
    lr=1e-4,
    recon_loss_weight=0.1,
    syncnet_loss_weight=0.02,
    contrastive_loss_weight=0.1,
    use_cross_attention=True,
    patch_sigreg_weight=5.0,
    gen_latent_sigreg_weight=5.0,
    direct_recon_loss_weight=1.0,
    perceptual_loss_weight=1.0,
    cond_dropout_prob=0.15,
    pooled_dropout_prob=0.0,
    audio_window_radius=5,
    reset_optimizer=False,
    checkpoint_dir="diffusion_checkpoints_genlatent",
    checkpoint_every_epochs=2,
    log_every_steps=50,
    resume_from=None,
):
    os.makedirs(checkpoint_dir, exist_ok=True)
    model.eval()  # frozen encoder throughout

    print(f"Building ConcatDataset over {len(train_data_roots)} training identities:")
    full_dataset, _per_identity_counts = build_concat_dataset(
        train_data_roots, seq_len=cfg.seq_len, audio_window_radius=audio_window_radius
    )
    val_size = max(1, int(len(full_dataset) * val_frac))
    train_size = len(full_dataset) - val_size
    train_dataset, val_dataset = random_split(
        full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(3072)
    )
    print(f"Dataset split: {train_size} train sequences, {val_size} in-distribution val sequences "
          f"(a small held-back slice of the TRAINING identities only - the fully held-out "
          f"identities are evaluated separately, see the decisive-check section).")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=min(batch_size, val_size), shuffle=True, num_workers=2)

    patch_projector = nn.Linear(768, patch_dim).to(device)
    windowed_audio_feature_dim = (2 * audio_window_radius + 1) * cfg.model.audio_feature_dim
    gen_latent_adapter = GenerationLatentAdapter(
        embed_dim=model.embed_dim, audio_feature_dim=windowed_audio_feature_dim, gen_dim=gen_dim
    ).to(device)
    mouth_content_projector = MouthContentProjector(patch_dim=patch_dim, contrastive_dim=model.embed_dim).to(device)
    GRID, MOUTH_PATCH_START = 14, 7 * 14
    diffusion_decoder = PatchTokenDiffusionDecoder(
        patch_dim=patch_dim, cond_dim=gen_dim, raw_audio_feature_dim=cfg.model.audio_feature_dim
    ).to(device)
    patch_pixel_decoder = PatchGridToImageDecoder(patch_dim=patch_dim).to(device)
    vgg_perceptual_loss = VGGPerceptualLoss(device=device).to(device)

    noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
    ddim_scheduler = DDIMScheduler(num_train_timesteps=1000)

    opt = torch.optim.AdamW(
        list(diffusion_decoder.parameters())
        + list(patch_pixel_decoder.parameters())
        + list(patch_projector.parameters())
        + list(gen_latent_adapter.parameters())
        + list(mouth_content_projector.parameters()),
        lr=lr,
    )

    start_epoch = 0
    if resume_from and os.path.exists(resume_from):
        print(f"Resuming decoder training from checkpoint: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        diffusion_decoder.load_state_dict(ckpt["diffusion_decoder_state_dict"], strict=False)
        patch_pixel_decoder.load_state_dict(ckpt["patch_pixel_decoder_state_dict"])
        patch_projector.load_state_dict(ckpt["patch_projector_state_dict"])
        _own_state = gen_latent_adapter.state_dict()
        _loaded, _skipped = [], []
        for _k, _v in ckpt["gen_latent_adapter_state_dict"].items():
            if _k in _own_state and _own_state[_k].shape == _v.shape:
                _own_state[_k].copy_(_v)
                _loaded.append(_k)
            else:
                _skipped.append(_k)
        gen_latent_adapter.load_state_dict(_own_state)
        print(f"  gen_latent_adapter: loaded {len(_loaded)} params, kept fresh init for {_skipped}.")
        if "mouth_content_projector_state_dict" in ckpt:
            mouth_content_projector.load_state_dict(ckpt["mouth_content_projector_state_dict"])
            print("  mouth_content_projector: loaded from checkpoint.")
        else:
            print("  mouth_content_projector: not in this checkpoint (new module) - fresh init.")
        if not reset_optimizer:
            opt.load_state_dict(ckpt["optimizer_state_dict"])
        else:
            print("reset_optimizer=True: fresh optimizer.")
        start_epoch = ckpt["epoch"] + 1
        print(f"Resuming from epoch {start_epoch}")

    writer = SummaryWriter(f"runs/diffusion_decoder_multi_identity_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    global_step = 0

    def encode_batch(frames, audio):
        frames = frames.to(device)
        audio = audio.to(device)
        B, T, C, H, W = frames.shape
        flat = rearrange(frames, "b t c h w -> (b t) c h w")
        flat_audio = rearrange(audio, "b t d -> (b t) d")

        with torch.no_grad():  # ONLY the frozen encoder forward pass
            enc_out = model.jepa.encoder(flat, interpolate_pos_encoding=True)
            tokens = enc_out.last_hidden_state
            cls_tok = tokens[:, 0]
            patch_tok_raw = tokens[:, 1:]
            pooled_state = model.jepa.projector(cls_tok)

        patch_tok = patch_projector(patch_tok_raw)
        if gen_latent_adapter.training and pooled_dropout_prob > 0:
            pooled_drop_mask = (
                torch.rand(pooled_state.size(0), device=device) < pooled_dropout_prob
            ).unsqueeze(-1)
            pooled_state_in = torch.where(
                pooled_drop_mask,
                gen_latent_adapter.null_pooled_state.unsqueeze(0).expand_as(pooled_state),
                pooled_state,
            )
        else:
            pooled_state_in = pooled_state
        real_gen_latent = gen_latent_adapter(pooled_state_in, flat_audio)

        if gen_latent_adapter.training and cond_dropout_prob > 0:
            drop_mask = (torch.rand(real_gen_latent.size(0), device=device) < cond_dropout_prob).unsqueeze(-1)
            gen_latent = torch.where(
                drop_mask, gen_latent_adapter.null_cond.unsqueeze(0).expand_as(real_gen_latent), real_gen_latent
            )
        else:
            gen_latent = real_gen_latent

        audio_embed = gen_latent_adapter.audio_proj(flat_audio)
        audio_seq = flat_audio.view(flat_audio.size(0), 2 * audio_window_radius + 1, cfg.model.audio_feature_dim)

        return flat, pooled_state, patch_tok, gen_latent, real_gen_latent, audio_embed, audio_seq, B, T

    @torch.no_grad()
    def cross_frame_ratio(flat, patch_tok, gen_latent, audio_embed, audio_seq, num_show=8, low_noise_timestep=50):
        n = min(num_show, flat.size(0))
        idx = torch.randperm(flat.size(0))[:n]
        gt, tok, cond, cond_audio, cond_audio_seq = flat[idx], patch_tok[idx], gen_latent[idx], audio_embed[idx], audio_seq[idx]

        t_low = torch.full((n,), low_noise_timestep, device=device, dtype=torch.long)
        noise = torch.randn_like(tok)
        noisy_low = noise_scheduler.add_noise(tok, noise, t_low)
        pred_noise_low = diffusion_decoder(noisy_low, t_low, cond, audio_embed=cond_audio,
                                            audio_seq=(cond_audio_seq if use_cross_attention else None))
        alpha_low = noise_scheduler.alphas_cumprod[t_low].to(device).view(-1, 1, 1)
        denoised_low = (noisy_low - (1 - alpha_low).sqrt() * pred_noise_low) / alpha_low.sqrt()
        decoded_low = patch_pixel_decoder(denoised_low)

        def mean_pairwise_l1(x):
            total, count = 0.0, 0
            for i in range(x.size(0)):
                for j in range(i + 1, x.size(0)):
                    total += (x[i] - x[j]).abs().mean().item()
                    count += 1
            return total / max(count, 1)

        gt_pw = mean_pairwise_l1(gt)
        low_pw = mean_pairwise_l1(decoded_low)
        return (low_pw / gt_pw) if gt_pw > 0 else float("nan")

    def sync_window_loss(flat, patch_tok, gen_latent, audio_embed, audio_seq, B, T, num_frames=5):
        if syncnet_loss_weight <= 0:
            return torch.tensor(0.0, device=device)
        gt_frames = rearrange(flat, "(b t) c h w -> b t c h w", b=B, t=T)[:, :num_frames]
        tok_win = rearrange(patch_tok, "(b t) n d -> b t n d", b=B, t=T)[:, :num_frames]
        cond_win = rearrange(gen_latent, "(b t) d -> b t d", b=B, t=T)[:, :num_frames]
        audio_win = rearrange(audio_embed, "(b t) d -> b t d", b=B, t=T)[:, :num_frames]
        audio_seq_win = rearrange(audio_seq, "(b t) w d -> b t w d", b=B, t=T)[:, :num_frames]
        tok_win = rearrange(tok_win, "b t n d -> (b t) n d")
        cond_win = rearrange(cond_win, "b t d -> (b t) d")
        audio_win = rearrange(audio_win, "b t d -> (b t) d")
        audio_seq_win = rearrange(audio_seq_win, "b t w d -> (b t) w d")

        t_low = torch.full((tok_win.size(0),), 50, device=device, dtype=torch.long)
        noise = torch.randn_like(tok_win)
        noisy_low = noise_scheduler.add_noise(tok_win, noise, t_low)
        pred_noise_low = diffusion_decoder(noisy_low, t_low, cond_win, audio_embed=audio_win,
                                            audio_seq=(audio_seq_win if use_cross_attention else None))
        alpha_low = noise_scheduler.alphas_cumprod[t_low].to(device).view(-1, 1, 1)
        denoised_low = (noisy_low - (1 - alpha_low).sqrt() * pred_noise_low) / alpha_low.sqrt()
        decoded_flat = patch_pixel_decoder(denoised_low)
        decoded_window = rearrange(decoded_flat, "(b t) c h w -> b t c h w", b=B, t=num_frames)

        return syncnet_perceptual_loss(decoded_window, gt_frames, num_frames=num_frames)

    for epoch in range(start_epoch, start_epoch + num_epochs):
        diffusion_decoder.train()
        patch_pixel_decoder.train()
        patch_projector.train()
        gen_latent_adapter.train()

        epoch_diff_loss, epoch_sync_loss, epoch_sigreg_loss, epoch_gen_latent_sigreg_loss, n_batches = 0.0, 0.0, 0.0, 0.0, 0
        epoch_direct_recon_loss, epoch_perceptual_loss, epoch_contrastive_loss = 0.0, 0.0, 0.0
        pbar = tqdm(train_loader, desc=f"Diffusion decoder epoch {epoch+1}/{start_epoch + num_epochs}")
        for frames, audio in pbar:
            flat_frames, pooled_state, patch_tok, gen_latent, real_gen_latent, audio_embed, audio_seq, B, T = encode_batch(frames, audio)

            opt.zero_grad()
            noise = torch.randn_like(patch_tok)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (patch_tok.size(0),), device=device
            ).long()
            noisy_patch_tok = noise_scheduler.add_noise(patch_tok, noise, timesteps)

            pred_noise = diffusion_decoder(noisy_patch_tok, timesteps, gen_latent, audio_embed=audio_embed,
                                            audio_seq=(audio_seq if use_cross_attention else None))
            diff_loss = F.mse_loss(pred_noise, noise)

            sync_loss = sync_window_loss(flat_frames, patch_tok, gen_latent, audio_embed, audio_seq, B, T)

            mouth_embed = mouth_content_projector(patch_tok[:, MOUTH_PATCH_START:, :])
            contrastive_loss = audio_content_contrastive_loss(mouth_embed, audio_embed)

            patch_tok_pooled = patch_tok.mean(dim=1)
            sigreg_loss = model.sigreg(patch_tok_pooled.unsqueeze(0))
            gen_latent_sigreg_loss = model.sigreg(real_gen_latent.unsqueeze(0))

            if direct_recon_loss_weight > 0 or perceptual_loss_weight > 0:
                direct_decoded = patch_pixel_decoder(patch_tok)
                direct_recon_loss = (F.mse_loss(direct_decoded, flat_frames)
                                      if direct_recon_loss_weight > 0 else torch.tensor(0.0, device=device))
                perceptual_loss = (vgg_perceptual_loss(direct_decoded, flat_frames)
                                    if perceptual_loss_weight > 0 else torch.tensor(0.0, device=device))
            else:
                direct_recon_loss = torch.tensor(0.0, device=device)
                perceptual_loss = torch.tensor(0.0, device=device)

            loss = (diff_loss + syncnet_loss_weight * sync_loss
                    + contrastive_loss_weight * contrastive_loss
                    + patch_sigreg_weight * sigreg_loss + gen_latent_sigreg_weight * gen_latent_sigreg_loss
                    + direct_recon_loss_weight * direct_recon_loss
                    + perceptual_loss_weight * perceptual_loss)

            loss.backward()
            opt.step()

            epoch_diff_loss += diff_loss.item()
            epoch_sync_loss += sync_loss.item()
            epoch_sigreg_loss += sigreg_loss.item()
            epoch_gen_latent_sigreg_loss += gen_latent_sigreg_loss.item()
            epoch_direct_recon_loss += direct_recon_loss.item()
            epoch_perceptual_loss += perceptual_loss.item()
            epoch_contrastive_loss += contrastive_loss.item()
            n_batches += 1

            if global_step % log_every_steps == 0:
                writer.add_scalar("train/diffusion_loss", diff_loss.item(), global_step)
                writer.add_scalar("train/syncnet_loss", sync_loss.item(), global_step)
                writer.add_scalar("train/patch_sigreg_loss", sigreg_loss.item(), global_step)
                writer.add_scalar("train/gen_latent_sigreg_loss", gen_latent_sigreg_loss.item(), global_step)
                writer.add_scalar("train/direct_recon_loss", direct_recon_loss.item(), global_step)
                writer.add_scalar("train/perceptual_loss", perceptual_loss.item(), global_step)
                writer.add_scalar("train/contrastive_loss", contrastive_loss.item(), global_step)
            global_step += 1
            pbar.set_postfix({"diff": f"{diff_loss.item():.4f}",
                               "sync": f"{sync_loss.item():.4f}", "sigreg": f"{sigreg_loss.item():.4f}",
                               "gen_latent_sigreg": f"{gen_latent_sigreg_loss.item():.4f}",
                               "direct_recon": f"{direct_recon_loss.item():.4f}", "perceptual": f"{perceptual_loss.item():.4f}",
                               "contrastive": f"{contrastive_loss.item():.4f}"})

        avg_diff = epoch_diff_loss / max(n_batches, 1)
        avg_sync = epoch_sync_loss / max(n_batches, 1)
        avg_sigreg = epoch_sigreg_loss / max(n_batches, 1)
        avg_gen_latent_sigreg = epoch_gen_latent_sigreg_loss / max(n_batches, 1)
        avg_direct_recon = epoch_direct_recon_loss / max(n_batches, 1)
        avg_perceptual = epoch_perceptual_loss / max(n_batches, 1)
        avg_contrastive = epoch_contrastive_loss / max(n_batches, 1)
        print(f"\nEpoch {epoch+1} train avg: diffusion_loss={avg_diff:.4f}  "
              f"syncnet_loss={avg_sync:.4f}  patch_sigreg_loss={avg_sigreg:.4f}  gen_latent_sigreg={avg_gen_latent_sigreg:.4f}  "
              f"direct_recon_loss={avg_direct_recon:.4f}  perceptual_loss={avg_perceptual:.4f}  contrastive_loss={avg_contrastive:.4f}"
              )
        writer.add_scalar("train_epoch/diffusion_loss", avg_diff, epoch)
        writer.add_scalar("train_epoch/syncnet_loss", avg_sync, epoch)
        writer.add_scalar("train_epoch/patch_sigreg_loss", avg_sigreg, epoch)
        writer.add_scalar("train_epoch/gen_latent_sigreg_loss", avg_gen_latent_sigreg, epoch)
        writer.add_scalar("train_epoch/direct_recon_loss", avg_direct_recon, epoch)
        writer.add_scalar("train_epoch/perceptual_loss", avg_perceptual, epoch)
        writer.add_scalar("train_epoch/contrastive_loss", avg_contrastive, epoch)

        diffusion_decoder.eval()
        patch_pixel_decoder.eval()
        patch_projector.eval()
        gen_latent_adapter.eval()
        val_diff_loss, val_batches = 0.0, 0
        val_ratio_sum, val_ratio_batches = 0.0, 0
        with torch.no_grad():
            for frames, audio in val_loader:
                flat_frames, pooled_state, patch_tok, gen_latent, real_gen_latent, audio_embed, audio_seq, B, T = encode_batch(frames, audio)

                noise = torch.randn_like(patch_tok)
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps, (patch_tok.size(0),), device=device
                ).long()
                noisy_patch_tok = noise_scheduler.add_noise(patch_tok, noise, timesteps)
                pred_noise = diffusion_decoder(noisy_patch_tok, timesteps, gen_latent, audio_embed=audio_embed,
                                                audio_seq=(audio_seq if use_cross_attention else None))
                diff_loss = F.mse_loss(pred_noise, noise)

                val_diff_loss += diff_loss.item()
                val_batches += 1

                batch_ratio = cross_frame_ratio(flat_frames, patch_tok, gen_latent, audio_embed, audio_seq)
                if batch_ratio == batch_ratio:
                    val_ratio_sum += batch_ratio
                    val_ratio_batches += 1

        avg_val_diff = val_diff_loss / max(val_batches, 1)
        ratio = val_ratio_sum / max(val_ratio_batches, 1) if val_ratio_batches else float("nan")
        print(f"\nEpoch {epoch+1} val avg (in-distribution):   diffusion_loss={avg_val_diff:.4f}  "
              f"cross_frame_ratio={ratio:.3f} (avg over {val_ratio_batches} val batches)")
        writer.add_scalar("val_epoch/diffusion_loss", avg_val_diff, epoch)
        writer.add_scalar("val_epoch/cross_frame_ratio", ratio, epoch)

        if (epoch + 1) % checkpoint_every_epochs == 0 or epoch == start_epoch + num_epochs - 1:
            ckpt_path = os.path.join(checkpoint_dir, f"diffusion_decoder_genlatent_epoch_{epoch+1}.pth")
            torch.save({
                "epoch": epoch,
                "diffusion_decoder_state_dict": diffusion_decoder.state_dict(),
                "patch_pixel_decoder_state_dict": patch_pixel_decoder.state_dict(),
                "patch_projector_state_dict": patch_projector.state_dict(),
                "gen_latent_adapter_state_dict": gen_latent_adapter.state_dict(),
                "mouth_content_projector_state_dict": mouth_content_projector.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    writer.close()
    return diffusion_decoder, patch_pixel_decoder, patch_projector, gen_latent_adapter, noise_scheduler, ddim_scheduler


print("train_diffusion_decoder() defined (multi-identity ConcatDataset version).")


## 8. Run training

Same 2-phase curriculum as the validated single-identity recipe (foundation phase:
diffusion + SIGReg only; main phase: phases in SyncNet perceptual loss + direct
reconstruction + VGG perceptual loss). Epoch counts (2 + 8 = 10) are scaled down from
the single-identity run (15 + 45 = 60 epochs on ~6,000 frames) for roughly the same
total number of gradient steps against the ~10x larger dataset.

**Estimated runtime on L4** (derived from the last full single-identity run - 15+45
epochs took 7h48m - scaled by this dataset's step-count ratio): Phase 1 (2 epochs)
~2h, Phase 2 (8 epochs) ~12h, ~14h total. To keep any single Colab session within an
8-10h window (avoids losing a large chunk of progress to a session limit or
disconnect), Phase 2 is split into two 4-epoch halves below - run **Phase 1 + Phase
2a together in one sitting (~8h)**, then **Phase 2b separately later (~6h)**,
resuming from Phase 2a's last checkpoint.

`checkpoint_every_epochs=1` throughout (was 2) - caps any lost progress from a
disconnect at ~1 epoch (~1-1.5h) instead of ~2 epochs.

`use_cross_attention=False` and `contrastive_loss_weight=0.0` throughout, per the
decision to pause architecture work until the data-scale question is answered.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs


In [ ]:
# Phase 1 of 2 - foundation (diffusion + SIGReg only, no reconstruction/perceptual/sync
# losses yet - matches the curriculum pattern that fixed training instability in the
# single-identity retrain). ~2h on L4. Run this + Phase 2a below in the same sitting.
diffusion_decoder, patch_pixel_decoder, patch_projector, gen_latent_adapter, noise_scheduler, ddim_scheduler = train_diffusion_decoder(
    model, cfg, train_data_roots=TRAIN_IDENTITIES, num_epochs=2, batch_size=16,
    resume_from=None,
    audio_window_radius=5,
    use_cross_attention=False,
    contrastive_loss_weight=0.0,
    cond_dropout_prob=0.0,
    pooled_dropout_prob=0.0,
    syncnet_loss_weight=0.0,
    perceptual_loss_weight=0.0,
    direct_recon_loss_weight=0.0,
    patch_sigreg_weight=5.0, gen_latent_sigreg_weight=5.0,
    checkpoint_dir="diffusion_checkpoints_multi_identity_v1",
    checkpoint_every_epochs=1,
)


In [ ]:
# Phase 2a of 2 (epochs 3-6 of 10) - phases in reconstruction + SyncNet perceptual +
# VGG perceptual losses, continuing the SAME optimizer (reset_optimizer=False - valid
# since phase 1's optimizer already included every param group this phase needs too).
# ~6h on L4 - run right after Phase 1 above, in the same Colab sitting (~8h combined).
diffusion_decoder, patch_pixel_decoder, patch_projector, gen_latent_adapter, noise_scheduler, ddim_scheduler = train_diffusion_decoder(
    model, cfg, train_data_roots=TRAIN_IDENTITIES, num_epochs=4, batch_size=16,
    resume_from="diffusion_checkpoints_multi_identity_v1/diffusion_decoder_genlatent_epoch_2.pth",
    reset_optimizer=False,
    audio_window_radius=5,
    use_cross_attention=False,
    contrastive_loss_weight=0.0,
    cond_dropout_prob=0.0,
    pooled_dropout_prob=0.0,
    syncnet_loss_weight=0.02,
    perceptual_loss_weight=0.03,
    direct_recon_loss_weight=1.0,
    patch_sigreg_weight=5.0, gen_latent_sigreg_weight=5.0,
    checkpoint_dir="diffusion_checkpoints_multi_identity_v1",
    checkpoint_every_epochs=1,
)


### Phase 2b - separate session

Resume here later (new Colab session is fine - everything needed is reloaded from the
checkpoint). ~6h on L4, well under the 8-10h target on its own.

In [ ]:
# Phase 2b of 2 (epochs 7-10 of 10) - same recipe as Phase 2a, resuming from its last
# checkpoint. ~6h on L4 - safe to run as its own session.
diffusion_decoder, patch_pixel_decoder, patch_projector, gen_latent_adapter, noise_scheduler, ddim_scheduler = train_diffusion_decoder(
    model, cfg, train_data_roots=TRAIN_IDENTITIES, num_epochs=4, batch_size=16,
    resume_from="diffusion_checkpoints_multi_identity_v1/diffusion_decoder_genlatent_epoch_6.pth",
    reset_optimizer=False,
    audio_window_radius=5,
    use_cross_attention=False,
    contrastive_loss_weight=0.0,
    cond_dropout_prob=0.0,
    pooled_dropout_prob=0.0,
    syncnet_loss_weight=0.02,
    perceptual_loss_weight=0.03,
    direct_recon_loss_weight=1.0,
    patch_sigreg_weight=5.0, gen_latent_sigreg_weight=5.0,
    checkpoint_dir="diffusion_checkpoints_multi_identity_v1",
    checkpoint_every_epochs=1,
)


## 9. Decisive check: held-out identity generalization vs. in-distribution fit

Same token-space mouth-region matching methodology used for every prior decisive check
in this project (`full_ddim_token_space_mouth_matching_check` family): full DDIM sampling
conditioned on `gen_latent`/`audio_only`/`pooled_only`, then nearest-neighbor match in
mouth-region patch-token space against the true frame it should match, pooled over
several seeds against a known chance baseline of 1/n per seed.

Run separately on:
- **`TRAIN_IDENTITIES`** (a fresh sample, not the exact frames trained on, but from
  identities the model DID see during training) - are gen_latent/audio_only still
  matching above chance on data it has some exposure to?
- **`HELD_OUT_IDENTITIES`** (AnnMarieBuerkle + DebFischer0, never seen during training
  at all) - the real test of whether more data produces something that generalizes,
  vs. just memorizing the training identities better.

If `audio_only` on `HELD_OUT_IDENTITIES` is meaningfully above chance, that is real
evidence the data-scale hypothesis is paying off. If it stays at chance while
`TRAIN_IDENTITIES` improves, the model is fitting the training set without learning a
transferable audio-to-mouth mapping - a different (and less encouraging) outcome that
would point back toward the objective/architecture, not data volume, as the bottleneck.

In [ ]:
# Load the just-trained (or previously saved) checkpoint fresh from disk, exactly like
# every prior decisive check in this project - avoids relying on in-memory state from
# the training cells above, so this cell is also safe to re-run standalone later.
_ckpt_path = "diffusion_checkpoints_multi_identity_v1/diffusion_decoder_genlatent_epoch_10.pth"  # adjust to your last checkpoint
_ckpt = torch.load(_ckpt_path, map_location=device)

patch_dim, gen_dim = 128, 256
audio_window_radius = 5
raw_audio_feature_dim = cfg.model.audio_feature_dim
windowed_audio_feature_dim = (2 * audio_window_radius + 1) * raw_audio_feature_dim

patch_projector = nn.Linear(768, patch_dim).to(device)
gen_latent_adapter = GenerationLatentAdapter(
    embed_dim=model.embed_dim, audio_feature_dim=windowed_audio_feature_dim, gen_dim=gen_dim
).to(device)
diffusion_decoder = PatchTokenDiffusionDecoder(
    patch_dim=patch_dim, cond_dim=gen_dim, raw_audio_feature_dim=raw_audio_feature_dim
).to(device)
patch_pixel_decoder = PatchGridToImageDecoder(patch_dim=patch_dim).to(device)

diffusion_decoder.load_state_dict(_ckpt["diffusion_decoder_state_dict"])
patch_pixel_decoder.load_state_dict(_ckpt["patch_pixel_decoder_state_dict"])
patch_projector.load_state_dict(_ckpt["patch_projector_state_dict"])
gen_latent_adapter.load_state_dict(_ckpt["gen_latent_adapter_state_dict"])
diffusion_decoder.eval(); patch_pixel_decoder.eval(); patch_projector.eval(); gen_latent_adapter.eval()

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
ddim_scheduler = DDIMScheduler(num_train_timesteps=1000)
USE_CROSS_ATTENTION_FOR_CHECK = False  # matches how this checkpoint was trained
print(f"Loaded checkpoint fresh from disk (epoch={_ckpt['epoch']+1}).")

GRID = 14
MOUTH_PATCH_START = 7 * GRID


@torch.no_grad()
def get_diagnostic_batch(dataset, num_frames=6, seed=0):
    """Real frames from `dataset` + their pooled_state/patch_tok/raw_audio."""
    gen = torch.Generator().manual_seed(seed)
    loader = DataLoader(dataset, batch_size=num_frames, shuffle=True, num_workers=2, generator=gen)
    frames, audio = next(iter(loader))
    frames, audio = frames.to(device), audio.to(device)
    B, T, C, H, W = frames.shape
    flat = rearrange(frames, "b t c h w -> (b t) c h w")
    raw_audio = rearrange(audio, "b t d -> (b t) d")

    enc_out = model.jepa.encoder(flat, interpolate_pos_encoding=True)
    tokens = enc_out.last_hidden_state
    pooled_state = model.jepa.projector(tokens[:, 0])
    patch_tok = patch_projector(tokens[:, 1:])

    n = flat.size(0)
    return flat, pooled_state, patch_tok, raw_audio, n


def _mean_pairwise_l2_flat(x):
    n = x.size(0)
    total, count = 0.0, 0
    for i in range(n):
        for j in range(i + 1, n):
            total += (x[i] - x[j]).norm().item()
            count += 1
    return total / max(count, 1)


@torch.no_grad()
def full_ddim_token_space_mouth_matching_check(dataset, vary, num_frames=6, ddim_steps=50,
                                                noise_seed=1234, batch_seed=42, num_show=6):
    """Same methodology as every prior decisive check in this project. `dataset` is
    whichever ConcatDataset (TRAIN_IDENTITIES-sample or HELD_OUT_IDENTITIES) to draw
    the batch from."""
    diffusion_decoder.eval()
    gen_latent_adapter.eval()

    flat, pooled_state, patch_tok, raw_audio, n = get_diagnostic_batch(dataset, num_frames=num_frames, seed=batch_seed)
    n = min(num_show, n)
    true_tok = patch_tok[:n]

    fixed_pooled = pooled_state[0:1].repeat(n, 1)
    fixed_audio = raw_audio[0:1].repeat(n, 1)

    if vary == "gen_latent":
        cond = gen_latent_adapter(pooled_state[:n], raw_audio[:n])
        audio_embed = gen_latent_adapter.audio_proj(raw_audio[:n])
        audio_seq = raw_audio[:n].view(n, 2 * audio_window_radius + 1, raw_audio_feature_dim)
    elif vary == "audio_only":
        cond = gen_latent_adapter(fixed_pooled, raw_audio[:n])
        audio_embed = gen_latent_adapter.audio_proj(raw_audio[:n])
        audio_seq = raw_audio[:n].view(n, 2 * audio_window_radius + 1, raw_audio_feature_dim)
    elif vary == "pooled_only":
        cond = gen_latent_adapter(pooled_state[:n], fixed_audio)
        audio_embed = gen_latent_adapter.audio_proj(fixed_audio)
        audio_seq = fixed_audio.view(n, 2 * audio_window_radius + 1, raw_audio_feature_dim)
    else:
        raise ValueError(vary)

    gen = torch.Generator(device=device).manual_seed(noise_seed)
    fixed_initial_noise = torch.randn(true_tok[0:1].shape, generator=gen, device=device).repeat(n, 1, 1)
    ddim_scheduler.set_timesteps(ddim_steps)
    sample = fixed_initial_noise.clone()
    for t in ddim_scheduler.timesteps:
        t_batch = torch.full((n,), int(t), device=device, dtype=torch.long)
        pred_noise = diffusion_decoder(sample, t_batch, cond, audio_embed=audio_embed,
                                        audio_seq=(audio_seq if USE_CROSS_ATTENTION_FOR_CHECK else None))
        sample = ddim_scheduler.step(pred_noise, t, sample).prev_sample

    sample_mouth = sample[:, MOUTH_PATCH_START:, :]
    true_mouth = true_tok[:, MOUTH_PATCH_START:, :]
    sample_flat = sample_mouth.reshape(n, -1)
    true_flat = true_mouth.reshape(n, -1)

    dist = torch.zeros(n, n, device=device)
    for i in range(n):
        for j in range(n):
            dist[i, j] = (sample_flat[i] - true_flat[j]).norm()

    nearest = dist.argmin(dim=1)
    correct = (nearest == torch.arange(n, device=device)).sum().item()
    accuracy = correct / n
    chance = 1.0 / n

    true_pw = _mean_pairwise_l2_flat(true_flat)
    sample_pw = _mean_pairwise_l2_flat(sample_flat)
    ratio = sample_pw / true_pw if true_pw > 0 else float("nan")

    return {"accuracy": accuracy, "chance": chance, "correct": correct, "n": n, "ratio": ratio}


print("get_diagnostic_batch() / full_ddim_token_space_mouth_matching_check() defined.")


In [ ]:
# Build the two evaluation datasets. TRAIN_IDENTITIES here samples fresh batches from
# identities the model trained on (in-distribution check); HELD_OUT_IDENTITIES were
# NEVER touched by train_diffusion_decoder (generalization check).
print("In-distribution (train identities) dataset:")
train_ident_dataset, _ = build_concat_dataset(TRAIN_IDENTITIES, seq_len=cfg.seq_len, audio_window_radius=audio_window_radius)
print("\nHeld-out (never trained on) dataset:")
held_out_dataset, _ = build_concat_dataset(HELD_OUT_IDENTITIES, seq_len=cfg.seq_len, audio_window_radius=audio_window_radius)

SEEDS = [(42, 50), (7, 50), (99, 50), (2024, 150)]  # same 4-seed pooling scheme used throughout this project

for label, dataset in [("TRAIN identities (in-distribution)", train_ident_dataset),
                        ("HELD-OUT identities (generalization)", held_out_dataset)]:
    print(f"\n=== {label} ===")
    results = {"gen_latent": [], "audio_only": [], "pooled_only": []}
    for seed, n in SEEDS:
        print(f"--- seed={seed}, n={n} ---")
        for vary in ["gen_latent", "audio_only", "pooled_only"]:
            r = full_ddim_token_space_mouth_matching_check(dataset, vary, num_frames=n, num_show=n, batch_seed=seed)
            print(f"  {vary:12s} correct={r['correct']}/{r['n']}  chance={r['chance']:.4f}  ratio={r['ratio']:.4f}")
            results[vary].append(r["correct"])

    print(f"\n--- {label}: pooled across all {len(SEEDS)} batches (expected total = 4 per variant under pure chance) ---")
    for vary in ["gen_latent", "audio_only", "pooled_only"]:
        total = sum(results[vary])
        print(f"{vary:12s} total correct = {total}  (per-batch: {results[vary]}, expected = 4)")

print("\nCompare HELD-OUT audio_only's total against this week's single-identity baselines "
      "(same 4-seed scheme, chance=4): buggy-data=5, data-fix-only=7 (best so far, z=1.50), "
      "curriculum-retrain=6, Phase3(null)=4, Phase4/4a/4b=3-5, Phase5(from-scratch+xattn+contrastive)=1 "
      "(worst so far). A real data-scale win = HELD-OUT audio_only meaningfully above 7.")
